In [2]:
import pandas as pd

# Cargar dataset principal
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race",
    "sex", "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

adult = pd.read_csv(url, names=columns, skipinitialspace=True)

# Cargar dataset auxiliar externo
external = pd.read_csv("external_dataset.csv")

print(adult.head())
print(external.head())

   age         workclass  fnlwgt  education  education_num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   
3   53           Private  234721       11th              7   
4   28           Private  338409  Bachelors             13   

       marital_status         occupation   relationship   race     sex  \
0       Never-married       Adm-clerical  Not-in-family  White    Male   
1  Married-civ-spouse    Exec-managerial        Husband  White    Male   
2            Divorced  Handlers-cleaners  Not-in-family  White    Male   
3  Married-civ-spouse  Handlers-cleaners        Husband  Black    Male   
4  Married-civ-spouse     Prof-specialty           Wife  Black  Female   

   capital_gain  capital_loss  hours_per_week native_country income  
0          2174             0              40  United-States  <=50K  
1             0             0             

## Clasificación de atributos del dataset Adult Census Income y posibles técnicas de anonimización

Clasificación de atributos del dataset Adult Census Income y posibles técnicas de anonimización

La anonimización de datos persigue reducir el riesgo de reidentificación de los individuos en un conjunto de datos, permitiendo al mismo tiempo conservar parte de su utilidad analítica. Para ello, es necesario distinguir entre identificadores directos, cuasi-identificadores y atributos sensibles, tal como se establece en la literatura clásica sobre privacidad estadística.

En el dataset Adult Census Income, que contiene información demográfica y socioeconómica utilizada para predecir si una persona percibe ingresos superiores o inferiores a 50.000 dólares anuales, los atributos pueden clasificarse del siguiente modo.

1. IDENTIFICADORES DIRECTOS

Los identificadores directos son aquellos atributos que permiten reconocer de forma inmediata a una persona sin necesidad de combinar información adicional. Algunos ejemplos habituales serían:

- nombre y apellidos
- número de pasaporte o DNI
- correo electrónico
- número de teléfono

En el dataset Adult Census Income no aparecen identificadores directos explícitos, ya que fue publicado con fines de investigación y sin datos personales evidentes. No obstante, el atributo 'fnlwgt' podría considerarse un identificador interno débil en ciertos contextos, ya que representa un peso estadístico asociado a cada registro, aunque su finalidad original no es identificar individuos.

Posibles técnicas de anonimización para identificadores directos

Las técnicas más comunes para este tipo de atributos son:

- Supresión: eliminación completa del atributo.
- Pseudonimización: sustitución por un código artificial.
- Hash criptográfico: transformación irreversible mediante funciones hash.
- Tokenización: reemplazo por valores equivalentes gestionados en una tabla segura.

Estas técnicas son adecuadas porque los identificadores directos no suelen ser necesarios para el análisis estadístico posterior.

2. Cuasi-identificadores

Los cuasi-identificadores son atributos que individualmente no identifican a una persona, pero cuya combinación puede permitir su reidentificación al cruzarlos con otras fuentes externas. Este problema fue demostrado por Latanya Sweeney al mostrar que atributos demográficos aparentemente inocuos podían identificar a individuos concretos.

En este dataset, los principales cuasi-identificadores son:

age
sex
native_country

También podrían considerarse cuasi-identificadores secundarios:

race
education
marital_status
occupation
hours_per_week
Justificación

Estos atributos se consideran cuasi-identificadores porque:

age reduce significativamente el conjunto de posibles individuos.
sex divide la población en grupos más pequeños.
native_country restringe aún más el número de coincidencias.
Su combinación puede hacer que un registro resulte único dentro del dataset.

Por ejemplo, la combinación:

age = 47
sex = Female
native_country = Japan

podría corresponder a muy pocos registros, facilitando un ataque de enlace con una base externa.

Técnicas de anonimización aplicables a cuasi-identificadores
Para age

Se pueden aplicar varias técnicas:

- Generalización por rangos: 27 → 20–29
- Redondeo: 27 → 30
- Supresión parcial: eliminar edades extremas
- Perturbación:añadir pequeño ruido aleatorio

La generalización por intervalos suele ser la más adecuada porque mantiene utilidad estadística.

Para sex

Posibles técnicas:

- Supresión
Male/Female → *
- Generalización
Male/Female → Person
- Agrupación parcial
mantener solo categorías amplias

La supresión suele emplearse cuando este atributo aporta poco valor analítico.

Para native_country

Técnicas posibles:

- Generalización geográfica
Spain → Europe
Mexico → America
- Agrupación regiona
l
Asia / Europe / America
- Supresión
Unknown

La generalización por regiones permite reducir la singularidad sin perder completamente la información geográfica.

3. Atributos sensibles

Los atributos sensibles contienen información cuya asociación con una persona debe protegerse especialmente.

En el dataset Adult Census Income, el atributo sensible principal es:

income

con valores:

<=50K
>50K

Este atributo se considera sensible porque revela información económica privada del individuo. La documentación oficial del dataset lo define como la variable objetivo del análisis.

Otros atributos que podrían considerarse sensibles dependiendo del contexto son:

race
occupation

# Función de anonimización para garantizar k-anonymity (k = 3)

Técnicas aplicadas
1. age → Generalización por intervalos

Se sustituye la edad exacta por rangos de 10 años.

Ejemplo
27 → 20-29
34 → 30-39
Justificación

La edad exacta puede hacer únicos muchos registros. La generalización reduce granularidad, pero conserva utilidad estadística.

2. sex → Supresión total

Se reemplaza por: '*'

Justificación

Este atributo tiene pocas categorías y aporta poco valor analítico en este ejercicio. Su supresión elimina capacidad de reidentificación.

3. native_country → Generalización geográfica

Los países se agrupan por regiones.

Ejemplo
Spain → Europe
Mexico → America
India → Asia
Justificación

Un país concreto puede identificar individuos poco frecuentes. La agrupación regional mantiene información geográfica general.

In [47]:
import pandas as pd

def generalize_country(country):
    europe = ['England', 'Germany', 'France', 'Italy', 'Portugal', 'Greece']
    america = ['United-States', 'Canada', 'Mexico', 'Cuba']
    asia = ['India', 'China', 'Japan', 'Iran', 'Philippines']
    
    if country in europe:
        return 'Europe'
    elif country in america:
        return 'America'
    elif country in asia:
        return 'Asia'
    else:
        return 'Other'
    
def generalize_age(x):
    if x >= 70:
        return "70+"
    lower = (x // 10) * 10
    return f"{lower}-{lower+9}"

def anonymize_k3(df):
    anon = df.copy()

    anon['age'] = anon['age'].apply(generalize_age)
    # 2. supresión
    anon['sex'] = '*'

    # 3. continente
    anon['native_country'] = anon['native_country'].apply(generalize_country)

    return anon


Aplicamos al dataset

In [48]:
adult_anon = anonymize_k3(adult)
print(adult_anon[['age','sex','native_country']].head())

     age sex native_country
0  30-39   *        America
1  50-59   *        America
2  30-39   *        America
3  50-59   *        America
4  20-29   *        America


Verificacion de k-anonymity




In [49]:
qi = ['age', 'sex', 'native_country']

groups = adult_anon.groupby(qi).size().reset_index(name='count')

print(groups.sort_values('count').head())

print("¿Cumple 3-anonymity?:", (groups['count'] >= 3).all())

      age sex native_country  count
2   10-19   *         Europe      6
25    70+   *           Asia      6
26    70+   *         Europe      8
1   10-19   *           Asia      9
21  60-69   *           Asia     24
¿Cumple 3-anonymity?: True


Explicación del criterio

La verificación comprueba que para cada combinación:

(age, sex, native_country)

existen al menos:

count >= 3

lo que significa que ningún individuo puede distinguirse de menos de otros dos registros.

Justificación metodológica

La combinación de técnicas elegida busca equilibrar:

✅ privacidad
✅ utilidad de los datos
✅ simplicidad de implementación

porque:

age conserva valor analítico mediante intervalos
sex elimina poca utilidad pero reduce riesgo
native_country mantiene región sin exponer país exacto

Este enfoque es coherente con las recomendaciones habituales de anonimización por generalización y supresión en modelos de k-anonymity.

## Comprobación de que anonimizar únicamente un atributo no garantiza k-anonymity

El modelo de **k-anonymity** exige que cada combinación de los cuasi-identificadores seleccionados aparezca al menos \(k\) veces en el conjunto de datos. En este ejercicio se ha fijado:

\[
k = 3
\]

Por tanto, cada combinación de:

- `age`
- `sex`
- `native_country`

debe aparecer al menos en 3 registros distintos.

Para comprobar si anonimizar únicamente un atributo es suficiente, se realizan dos pruebas independientes:
1. anonimizar solo `age`
2. anonimizar solo `native_country`

---

### 1. Anonimización únicamente de `age`

En esta prueba se generaliza la edad en intervalos de 10 años, manteniendo sin cambios `sex` y `native_country`.


In [50]:
adult_age = adult.copy()

# Generalizar solo age en intervalos de 10 años

adult_age['age'] = adult_age['age'].apply(generalize_age)

# Calcular tamaño de cada grupo
groups_age = adult_age.groupby(
    ['age', 'sex', 'native_country']
).size().reset_index(name='count')

# Mostrar grupos más pequeños
print(groups_age.sort_values('count').head(10))

# Verificar k-anonymity
print("¿Cumple k=3?:", (groups_age['count'] >= 3).all())

       age     sex native_country  count
360  60-69  Female          Haiti      1
361  60-69  Female           Hong      1
263  40-49    Male           Laos      1
363  60-69  Female           Iran      1
365  60-69  Female        Jamaica      1
366  60-69  Female          Japan      1
367  60-69  Female         Mexico      1
368  60-69  Female      Nicaragua      1
256  40-49    Male        Hungary      1
370  60-69  Female    Philippines      1
¿Cumple k=3?: False


In [51]:


adult_country = adult.copy()

# Generalizar solo native_country
adult_country['native_country'] = adult_country['native_country'].apply(generalize_country)

# Calcular tamaño de cada grupo
groups_country = adult_country.groupby(
    ['age', 'sex', 'native_country']
).size().reset_index(name='count')

# Mostrar grupos más pequeños
print(groups_country.sort_values('count').head(10))

# Verificar k-anonymity
print("¿Cumple k=3?:", (groups_country['count'] >= 3).all())

     age     sex native_country  count
342   61  Female           Asia      1
416   73    Male           Asia      1
434   78  Female          Other      1
449   83  Female        America      1
210   44  Female           Asia      1
343   61  Female         Europe      1
421   74    Male         Europe      1
422   74    Male          Other      1
43    23  Female           Asia      1
319   58  Female           Asia      1
¿Cumple k=3?: False


Los resultados muestran que anonimizar un único cuasi-identificador no es suficiente para garantizar k-anonymity, ya que la reidentificación depende de la combinación conjunta de todos los cuasi-identificadores.

En este dataset, solo la anonimización simultánea de:

age
sex
native_country

permite reducir adecuadamente el riesgo de reidentificación y cumplir la condición:

## Ataque de enlace (*linkage attack*) utilizando un dataset externo

Aunque un dataset cumpla formalmente la propiedad de **k-anonymity**, sigue existiendo riesgo de reidentificación cuando un atacante dispone de información auxiliar externa.

En este caso, el atacante dispone de dos fuentes:

### Dataset anonimizado
Contiene:
- `age`
- `sex`
- `native_country`
- `income`

### Dataset externo (`external_dataset.csv`)
Contiene:
- `name`
- `passport_number`
- `age`
- `sex`
- `native_country`

El atacante puede cruzar ambos datasets mediante los cuasi-identificadores comunes:

- `age`
- `sex`
- `native_country`

Si una combinación de estos atributos aparece una sola vez tras el cruce, el atacante puede asociar de forma inequívoca:

- el nombre real del individuo
- su número de pasaporte
- su categoría salarial (`income`)

Este proceso se conoce como **ataque de enlace**, ya que combina información de varias fuentes para romper el anonimato del dataset publicado.

---

## Implementación del linkage attack

El siguiente script realiza el cruce entre el dataset externo y el dataset anonimizado.



In [53]:

# Cargar datasets
external = pd.read_csv("external_dataset.csv")

# Cruce usando cuasi-identificadores
merged = pd.merge(
    external,
    adult_anon,
    on=['age', 'sex', 'native_country'],
    how='inner'
)

# Contar coincidencias por persona
matches = merged.groupby(
    ['name', 'passport_number']
).size().reset_index(name='matches')

# Seleccionar únicamente coincidencias únicas
unique_matches = matches[matches['matches'] == 1]

# Obtener registros reidentificados
reidentified = pd.merge(
    unique_matches[['name', 'passport_number']],
    merged,
    on=['name', 'passport_number']
)

# Mostrar resultado final
result = reidentified[['name', 'passport_number', 'income']]
print(result)

ValueError: You are trying to merge on int64 and object columns for key 'age'. If you wish to proceed you should use pd.concat